# Shipping Delays Prediction
___

**Table of Contents:**

- Project Overview
- Dataset
- Loading data
- Data exploration and cleaning
- Feature selection and Engineering
- High-revenue clusters
- Data splitting
- Late Delivery Classification
- Delays Prediction

Student: Nicola Laterza

## Project Overview

This project aims to enhance the logistics operations of the shipping company by leveraging predictive analytics to forecast shipping delays.
The main task is to predict **shipping delays** and quantify the expected delay in days. The ultimate goal is to provide
better service to the most profitable customers by tailoring predictions and evaluations to prioritize **high-revenue** customers/products. 

This project will help improve customer satisfaction and operational efficiency by minimizing delays for high-revenue segments.
The final models will empower the company to make data-driven decisions, ultimately leading to better resource allocation and improved service quality.

To implement this project I will use PySpark library.

In [ ]:
pip install holidays

## Dataset

The dataset is made of 53 columns providing a comprehensive evaluation of the orders, items purchased and customers from Puerto Rico and USA (EE. UU.).

Some basic information about the original dataset:
- Format: csv
- Size: 93.5 MB
- Rows: 180519 (excluding the header row)
- Columns: 53 (excluding the progressive index)

## Loading data

In [ ]:
FILE_PATH = '/FileStore/tables/SupplyChainDataset_V2.csv'
display(dbutils.fs.ls(FILE_PATH))

In [ ]:
original_df = spark.read.csv(
    path=FILE_PATH,
    inferSchema=True,
    header=True
)

In this case I use inferSchema=True to automatically infer the schema because the dataset has a lot of columns, but then it is necessary to check the type. Alternatively, we could manually define the schema, indicating the type for each column.

## Data Exploration and Cleaning

In the next cell we can analyze the Data Profile of the dataset, with basic statistics of all features. In the other graphs we can see the percentage of the values of some categorical features. Note: I use Pie Charts to simply show the percentage values in the Visualization Editor (it doesn't allow to show percentages for simple bar charts).

*Note*: To display all the graphs in this notebook correctly, data should not be truncated.

In [ ]:
# Basic graphs
display(original_df)

**First insights**

From the graphs we can see that:
- **Delivery Status**: About half of the records are *Late deliveries* (54.8%) and we also have 4.3% *canceled shippings*
- **Customer Country**: 61.6% of the Customers are from EE. UU. (that is USA), while 38.4% are from Puerto Rico
- **Shipping Mode**: Most of the shippings are Standard Class (59.7%)
- **Market**: the 3 main destination markets are LATAM (i.e. Latin America), Europe and Pacific Asia. The minor markets are USCA, that is USA (14.3%) and Africa (6.4%)

**First data cleaning**

From the dataset and the Data Profile we can see that there are some useless columns, so the first thing to do is to remove them.

- **Anonymized** cols: Customer Email and Customer Password are uninformative and anonymized
- **Missing values**: Order Zipcode (86% null), Product Description (100% null). The other columns don't have missing values.
- **0 variance**: Product Status
- **Duplicate** columns: Order Customer Id (same as Customer Id), Product Category Id (same as Category Id)
- **Other useless features**: _c0, Customer Fname, Customer Lname, Product Image, Customer Street, Product Card Id, Department Id, Order Item Cardprod Id
- Since the Shipping Date will not be available for new orders whose delays we want to predict, we have to drop it.


The other columnns will be analyzed later.

Moreover, the two dates have a string format, but they must have a date type, so we have to fix the format.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import to_date, to_timestamp, expr, when, col, lit, month, year

In [ ]:
delete_columns = ("Customer Email", "Customer Password", "Order Zipcode", "Product Description", "Product Status", "Order Status", "Order Customer Id", "Product Category Id", "_c0", "Customer Fname", "Customer Lname", "Product Image", "Product Description", "Customer Street", "Product Card Id", "Department Id", "Order Item Cardprod Id", "shipping date (DateOrders)")

df = original_df.drop(*delete_columns)

# Fix data type
df = df.withColumn("Order Date", to_timestamp(col("order date (DateOrders)"), "M/d/yyyy H:mm")) \
       .drop("order date (DateOrders)")

### Collinearities

Let's analyze the correlation matrix between the numerical features

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

corr_mat = df.toPandas().corr()

plt.figure(figsize=(13, 11))
sns.heatmap(corr_mat, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.show()

We can see 1.0 correlation between:
- "Sales", "Sales per customer" and "Order Item Total"
- "Order Profit Per Order" and "Benefit per order"
- "Order Item Product Price" and "Product Price"

There is also a 0.79 correlation between "Sales" and "Product Price" because Sales = Product Price * Order Item Quantity.

So we can keep only "Order Item Total" (that is Sales - Order Item Discount, so it's the real sales from a single row of the dataset) and "Order Profit Per Order".

In [ ]:
df = df.drop("Sales per customer", "Sales", "Benefit per order", "Order Item Product Price", "Product Price")

## Feature Selection and Engineering

### Target variable

Since the goal is to predict shipping delays and quantify the expected delay in days, we also need a target variable that is the delta between “Days for shipping (real)” and “Days for shipment (scheduled)” columns and then we can drop these columns. But how to manage advance shippings and canceled shippings? These shippings don't have delays.

The choice of the target variable impacts the type of model that we want to develop. In particular, I have identified 3 main options:
1. Use a regression model to predit only delays (setting advance and canceled shippings to 0 days of delay). This leads to a loss of information and an umbalanced dataset, because lots of shipping will have 0 days of delay.
2. Use a regression model to predict both delays (positive delta) and advances (negative delta). This can increase model complexity and lose the focus on delays, that is the main goal of the project.
3. Use a classification model to predict Late deliveries and then a regression model (or eventually another classification model) to predict the days of delay for these deliveries. This method could help the model to focus better on late deliveries and differentiate insights and performance of the two models (prediction of delay days can be more inaccurate).

Moreover, the *canceled shippings* have a "Late_delivery_risk" equal to 0, but the delta between the "Days for shipping (real)" and "Days for shipment (scheduled)" have both positive and negative values. This is an inconsistency and the meaning of the first column is not clear in this case. This consideration causes problems with option 1 (canceled shippings, advanced shippings and shippings on time are treated in the same way) and with option 2 (canceled shippings are treated as if they were not canceled), so it makes more helpful to first identify the late delivery shippings.

Considering all this, I choose **option 3**, but it is a more complex option to implement.

***Note***: I initially considered this option also because in the first version of the provided dataset there weren't the columns related to days (real and *scheduled*) and it was not possible to predict the dalays in days. After some time, with the second version of the dataset, it became possible to predict the delay days for late delivery shippings.

In [ ]:
display(df.filter(df["Delivery Status"] == "Shipping canceled").limit(50))

In [ ]:
# Add the "Days late" column
df = df.withColumn("Days late", df['Days for shipping (real)'] - df['Days for shipment (scheduled)'])
display(df.limit(50))

Let's analyze some relations between Delivery Status (in particular Late delivery) and the other features

In [ ]:
display(df)

In [ ]:
display(df.withColumn("profit_class",
                   when(df["Order Item Profit Ratio"] < 0, "negative profit")
                   .when((df["Order Item Profit Ratio"] >= 0) & (df["Order Item Profit Ratio"] <= 0.3), "low profit")
                   .when((df["Order Item Profit Ratio"] > 0.3), "high profit")
) \
.withColumn("order_total_class",
                   when(df["Order Item Total"] < 120, "1")
                   .when((df["Order Item Total"] >= 120) & (df["Order Item Total"] < 240), "2")
                   .when((df["Order Item Total"] >= 240) & (df["Order Item Total"] < 350), "3")
                   .when(df["Order Item Total"] >= 350, "4")
)
)

We can see that:

- **Shipping Mode** is an important feature to predict Late deliveries. Specifically, almost all of the **First Class** shippings are Late deliveries (95%), followed by Second Class (77%) and Same Day (46%). The Standard Class shipping is the better group, with 38% late deliveries, 38% avdance shippings and 19% shippings on time.
- **Type**: the TRANSFER type has a lower relevant percentage of Late deliveries than the other and all of the canceled shippings have this payment type.
- The different Customer Segments, Customer Country, Market have a very similar distribution of Delivery Statuses
- Profit Classes or Order Total classes have a very similar avg late delivery risk

Other features:
- Regarding the destination of the shipping, Order City, Order State and Order Country are too specific and have lots of different values. It is better to use **Region** (23 values). For example, we can see that Canada has a lower Avg(late delivery risk).

**Dates**

Regarding the Order Date, we can decompose it in month, week day and hour and see if there are relevant patterns.

In [ ]:
df = df.withColumn("Weekday", F.date_format(F.col("Order Date"), "EEEE"))  \
       .withColumn("Month", month(df["Order Date"])) \
       .withColumn("Hour", F.hour(df["Order Date"])) \
       .withColumn("Year", F.year(df["Order Date"]))

display(df)

From the above graphs we can see that there is an interesting insight about the **hours**: orders made at or before 12:00 have a lower avg(late delivery risk). Maybe this can be attributed to the tendency of the logistics hub to operate primarily in the first hours. When orders are placed during peak activity hours, they can be processed quickly and dispatched immediately afterward.

I don't consider the Year feature, because we want to predict future shippings, not shippings of the same years

In [ ]:
from pyspark.sql.functions import col, when
df = df.withColumn("Before 12:00", when(col("Hour") < 12, 1).otherwise(0))
display(df)

**Holidays**

An useful feature is "isHoliday" because shippings made during holiday days can have a greater risk of Late Delivery

In [ ]:
import holidays
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

us_holidays = holidays.US()

def is_holiday(date):
    if (date in us_holidays):
        return 1
    else: return 0

# UDF function
is_holiday_udf = F.udf(lambda date: is_holiday(date), IntegerType())

df = df.withColumn('isHoliday', is_holiday_udf(F.to_date(F.col('Order Date'))))

display(df)

**Other possible features**

*Note*: a potentially useful feature could be the **distance** between the location of store (Latitude and Longitude provided) and the destination place. To obtain this feature we should first get Latitude and Longitude of the destination place from "Order Country" and "Order City" columns using libraries like geopy; and then we can compute the distance between the two pairs of coordinates. 
Because of the limitations of these libraries and some errors in the country and city names, this feature extraction is more complicated and I don't do it.

**Outliers**

Regarding outliers, the most relevant columns are 'Order Item Total', 'Order Profit Per Order', 'Order Item Discount'. Let's analyze them.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

pandas_df = df.select('Order Item Total', 'Order Profit Per Order', 'Order Item Discount').toPandas()

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
sns.boxplot(data=pandas_df, x='Order Item Total', ax=axes[0])
sns.boxplot(data=pandas_df, x='Order Profit Per Order', ax=axes[1])
sns.boxplot(data=pandas_df, x='Order Item Discount', ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
display(original_df.where(df["Order Item Total"] > 800).select("Order Item Total", "Delivery Status", "Category Name", "Product Name", "Order Item Quantity", "Order Item Profit Ratio", "Order Profit Per Order", ).orderBy("Order Item Total", ascending=False))

In [ ]:
display(original_df.where(df["Order Item Discount"] > 200).withColumn("Real Order Item Total", original_df["Sales"]-original_df["Order Item Discount"]).select("Order Item Discount", "Order Profit Per Order", "Sales", "Order Item Discount Rate", "Delivery Status", "Category Name", "Product Name", "Order Item Quantity", "Order Item Total", "Real Order Item Total" ).orderBy("Order Item Discount", ascending=False))

In [ ]:
display(original_df.where(df["Order Profit Per Order"] < -500).select( "Order Profit Per Order", "Order Item Total", "Order Item Discount", "Order Item Discount Rate", "Delivery Status", "Category Name", "Product Name", "Order Item Quantity" ).orderBy("Order Profit Per Order"))

Is this data wrong?
We can see that all shippings with "Order Item Total" > 800 contains products from "Strength Training", "Computers" or
"Basketball" categories, specifically "SOLE E25 Elliptical", "SOLE E35 Elliptical" and "Dell Laptop" which are indeed expensive products. So these values are not wrong.

"Order Item Discount" > 200 are not wrong because "Order Item Total" is exaclty "Sales" - "Order Item Discount".

Regarding 'Order Profit Per Order', we can see that there are some high negative values, but since the dataset only provide the selling price and not the cost price, I can't compute the real profit. So, for simplicity, I assume that these values are correct.

*Note*: since I'm not going to use these features in the model, I don't consider this issue very much.

**Summary Statistics (Data Profile)**

In [ ]:
df=df.drop("Days for shipping (real)", "Category Id", "Customer Id", "Customer Zipcode", "Latitude", "Longitude", "Order Id", "Order Item Id", "Year", "Customer City", "Order City", "Product Name", "Order State", "Order Country", "Order Item Discount", "Order Item Discount Rate", "Order Item Total")
display(df)

Regarding the categorical features, are they nominal or ordinal? They are all nominal.

*Note*: for a more accurate representation, the features related to dates could be considered as cyclic variables (with sin/cos)

---

## High-revenue clusters

To select the high-revenue clusters, we can consider some relevant features and aggregate the “Order Profit Per Order” or “Order Item Profit Ratio”, because they identify how profitable the transaction was. Specifically, we can aggregate the average profit ratio, the total profit or the average profit.
The total profit is affected by the different proportion of shippings in each cluster while the avg profit ratio is not a good indicator in this case. Instead, the average profit tells how much profitable was the shipping, without considering the amount of orders in that cluster.

What features to consider? There are lots of different features or combinations that we could use. As suggested in the project work assignment, we can consider clusters related to the items (e.g., Category Name) or to the Customer (e.g., Customer Segment).

In [ ]:
# Considering Customers Country and Segment
df1 = df.groupBy("Customer Country", "Customer Segment").agg(
    F.avg("Order Item Profit Ratio").alias("Avg Profit Ratio"),
    F.sum("Order Profit Per Order").alias("Tot Profit"),
    F.avg("Order Profit Per Order").alias("Avg Profit")
)
display( 
    df1.withColumn("% Tot Profit", df1["Tot Profit"] / 3966902).orderBy("Tot Profit", ascending=False) # segment profit / total profit
)

In [ ]:
# Considering Customer State
df1 = df.groupBy("Customer State").agg(
    F.avg("Order Item Profit Ratio").alias("Avg Profit Ratio"),
    F.sum("Order Profit Per Order").alias("Tot Profit"),
    F.avg("Order Profit Per Order").alias("Avg Profit")
)
df1 = df1.withColumn("% Tot Profit", df1["Tot Profit"] / 3966902).orderBy("Tot Profit", ascending=False)
display(df1)

In [ ]:
# Considering Category Name
df1 = df.groupBy("Category Name").agg(
    F.avg("Order Item Profit Ratio").alias("Avg Profit Ratio"),
    F.sum("Order Profit Per Order").alias("Tot Profit"),
    F.avg("Order Profit Per Order").alias("Avg Profit")
)
display(df1.orderBy("Avg Profit", ascending=False))

I choose the item-related clusters, based on "Category Name".

Now I'm going to create a weight column using "Avg Profit" that will be useful in tailoring the training and evaluation process.

In this way, the orders containing product categories with high average profit obtain more importance. For example, we can see that the "Computers" category has the higher avg profit, so they sholud be prioritized

In [ ]:
df1 = df1.select("Category Name", "Avg Profit")

# Left join
df = df.join(df1, on="Category Name", how="left").withColumnRenamed("Avg Profit", "weight")
display(df.limit(50))

## Data splitting

In this case I use an **Hold-out** split because we have enough data and to have lower training time. Of course, if we want to have a better validation strategy, we could use Cross-validation instead, but it requires a longer training time.

As suggested in the project work assignment, I also consider a **temporal splitting**, because it better reflects real-world behavior.
Since the model is going to be used to predict future shipping delays, the training occurs only on past data with respect of the data on which to make the prediction.

In [ ]:
from pyspark import StorageLevel

df_2 = df.orderBy("Order Date")

# Split (80-10-10) Training, Validation, Test
total_rows = df_2.count()
train_size = 0.8
validation_size = 0.1
test_size = 0.1

# Split indexes
train_index = int(train_size * total_rows)
validation_index = int((train_size + validation_size) * total_rows)

# Dataframe for Train, Validation, Test with temporal splitting
train_df = df_2.limit(train_index).persist(StorageLevel.MEMORY_AND_DISK) 
validation_df = df_2.limit(validation_index).subtract(train_df).persist(StorageLevel.MEMORY_AND_DISK) 
test_df = df_2.subtract(train_df).subtract(validation_df).persist(StorageLevel.MEMORY_AND_DISK)

In [ ]:
# Check
print("Train: ")
print(train_df.agg(F.min("Order Date").alias("min_date")).first()["min_date"])
print(train_df.agg(F.max("Order Date").alias("max_date")).first()["max_date"])
print("Validation: ")
print(validation_df.agg(F.min("Order Date").alias("min_date")).first()["min_date"])
print(validation_df.agg(F.max("Order Date").alias("max_date")).first()["max_date"])
print("Test: ")
print(test_df.agg(F.min("Order Date").alias("min_date")).first()["min_date"])
print(test_df.agg(F.max("Order Date").alias("max_date")).first()["max_date"])

In [ ]:
# Class values count

train_size = train_df.count()
val_size = validation_df.count()
test_size = test_df.count()

train_late = train_df.filter(train_df.Late_delivery_risk == 1).count()
val_late = validation_df.filter(validation_df.Late_delivery_risk == 1).count()
test_late = test_df.filter(test_df.Late_delivery_risk == 1).count()

train_percentage = (train_late / train_size) * 100
val_percentage = (val_late / val_size) * 100
test_percentage = (test_late / test_size) * 100

print(f"Training set size: {train_size} -> % Late Deliveries: {train_percentage:.2f}%")
print(f"Validation set size: {val_size} -> % Late Deliveries: {val_percentage:.2f}%")
print(f"Test set size: {test_size} -> % Late Deliveries: {test_percentage:.2f}%")

Class abundances in each set are very similar to each other and the two classes are balanced.

Now we can build our models for Late Delivery classification and then for delay days prediction. I will consider these models separately. After training and validation, in a real scenario, the second model should be applied to the shippings classified as Late Delivery by the first model.

## Late Delivery Classification

### RandomForest

To perform the Late Delivery binary classification, we can use for example a Random Forest Classifier

Here we are not encoding categorical variables because decision trees and random forest can handle categorical variables and also from a performance point of view it is better in this case to do so.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

# Categorical features
categorical_columns = ['Type', 'Shipping Mode', 'Weekday', 'Order Region', "Department Name"]

# Categorical columns -> numeric values (_indexed)
indexers = [
    StringIndexer(inputCol=column, outputCol=column + "_indexed").fit(df)
    for column in categorical_columns
]

# updated column names
assembler_input = ['Type_indexed', 'Shipping Mode_indexed', 'Weekday_indexed', 'Order Region_indexed', 'Department Name_indexed', 'Hour', 'isHoliday', 'Month']

# Combine all features in a single vector
assembler_output = "features_vector"
assembler = VectorAssembler(
    inputCols=assembler_input,
    outputCol=assembler_output
)

# Model
rf = RandomForestClassifier(featuresCol=assembler_output, labelCol='Late_delivery_risk', weightCol='weight')

# Pipeline
stages = indexers + [assembler, rf]
pipeline = Pipeline(stages=stages)

In [ ]:
# Fit the pipeline to check
model_1 = pipeline.fit(train_df)

In [ ]:
print(rf.explainParams())

**Validation**

To validate the model I use the **accuracy** metric (weighted using the computed weight col), because the dataset is balanced and there isn't a business indication regarding the cost of False Positive errors or False Negative errors.

*Note*: we could also use the Recall to retrieve the maximum number of Late deliveries, since the next step will be the prediction of delay days of Late delivery shippings.

Moreover, I use a **GridSearch** strategy to to try different combinations of hyperparameters. An alternative to GridBuilder can be RandomGridBuilder.

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# GridSearch
def grid_search(train_df, validation_df, paramGrid, pipeline, evaluator):
    best_model = None
    best_score = float("-inf")
    best_params = None
    
    for params in paramGrid:
        # apply params to pipeline
        pipeline_ = pipeline.copy(params)
        
        # Fit on train set
        fitted_pipeline = pipeline_.fit(train_df)
        
        # Evaluate on validation set
        predictions = fitted_pipeline.transform(validation_df)
        score = evaluator.evaluate(predictions)
        
        if score > best_score:
            best_score = score
            best_model = fitted_pipeline
            best_params = params

    recall = evaluator.setMetricName('recallByLabel').evaluate(predictions)
    return best_model, best_params, best_score, recall

In [ ]:
# Training without weightCol

from pyspark.ml.tuning import ParamGridBuilder

evaluator_nw = MulticlassClassificationEvaluator(labelCol="Late_delivery_risk", metricName="accuracy")

# hyperparameters grid
paramGrid = (
    ParamGridBuilder()
    .addGrid(rf.maxDepth, [5, 10])  #  (default 5)
    .addGrid(rf.numTrees, [20, 30])  # (default 20)
    .build()
)

# Execute the grid search
best_pipeline, best_params, best_score, recall = grid_search(train_df, validation_df, paramGrid, pipeline, evaluator_nw)

# Best hyperparameters and accuracy
print(best_params)
print(best_score)
print("Recall: ", recall)

In [ ]:
# Training with weightCol

from pyspark.ml.tuning import ParamGridBuilder

evaluator = MulticlassClassificationEvaluator(labelCol="Late_delivery_risk", metricName="accuracy", weightCol="weight")

# hyperparameters grid
paramGrid = (
    ParamGridBuilder()
    .addGrid(rf.maxDepth, [5, 10])  # (default 5)
    .addGrid(rf.numTrees, [20, 30])  # (default 20)
    .build()
)

# Execute the grid search
best_pipeline, best_params, best_score, recall = grid_search(train_df, validation_df, paramGrid, pipeline, evaluator)

# Best hyperparameters and accuracy
print(best_params)
print(best_score)
print("Recall: ", recall)

With maxDepth=5 and numTrees=30, I obtained 71.48% accuracy. The difference between the training with or without 
weights is very small.

We can see that the Recall is high (87.61%)

### GBT Classifier

We can try to improve the performance of the first model, using for example Gradient-Boosted Trees (GBTs) learning algorithm for classification. To lower the training time, I avoid the GridSearch in this case.

In [ ]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(labelCol="Late_delivery_risk", featuresCol=assembler_output, maxIter=10, maxDepth=5, weightCol='weight')

pipeline = Pipeline(stages = indexers + [assembler, gbt])

In [ ]:
pipeline_model = pipeline.fit(train_df)

validation_predictions = pipeline_model.transform(validation_df)

score = evaluator.evaluate(validation_predictions)
print(f"Accuracy on Validation Set: {score}")

recall = evaluator.setMetricName('recallByLabel').evaluate(validation_predictions)
print(f"Recall on Validation Set: {recall}")

The accuracy is very similar to the RandomForest, so it has not improved the performance.

**Test**

In [ ]:
# Test of the first model chosen - RandomForestClassifier On TEST set
pred_df = best_pipeline.transform(test_df)

evaluator = MulticlassClassificationEvaluator(labelCol="Late_delivery_risk", metricName="accuracy", weightCol="weight")

accuracy_w = evaluator.evaluate(pred_df)
f1_w = evaluator.setMetricName('f1').evaluate(pred_df)

print(f"Accuracy_w is {accuracy_w}")
print(f"F1_w is {f1_w}")

Accuracy = 72.2 %

## Delay prediction

For the prediction of delay days we can use a regression model, for example a RandomForestRegressor. For brevity, I don't perform the GridSearch (but of course we could try to improve the performance by using different models, a better hyperparameter tuning, etc.)

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Only late deliveries
train_late = train_df.filter(col("Late_delivery_risk") == 1)
validation_late = validation_df.filter(col("Late_delivery_risk") == 1)
test_late = test_df.filter(col("Late_delivery_risk") == 1)

categorical_columns = ['Type', 'Shipping Mode', 'Weekday', 'Order Region', 'Department Name', 'isHoliday']
numeric_columns = ['Hour', 'Month']

# Indexing and One-Hot Encoding for categorical features
indexers = [StringIndexer(inputCol=col, outputCol=col + "_index") for col in categorical_columns]
encoders = [OneHotEncoder(inputCol=col + "_index", outputCol=col + "_encoded") for col in categorical_columns]

# VectorAssembler
assembler_inputs = [col + "_encoded" for col in categorical_columns] + numeric_columns
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

In [ ]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

rf_regressor = RandomForestRegressor(featuresCol="features", labelCol="Days late", predictionCol="prediction_delay", weightCol='weight', maxDepth=10, numTrees=20)

# pipeline
pipeline_regressor = Pipeline(stages=indexers + encoders + [assembler, rf_regressor])

# Training on train set
model_regressor = pipeline_regressor.fit(train_late)

# Validation on validation set
predictions_delay = model_regressor.transform(validation_late)

# RMSE
regression_evaluator = RegressionEvaluator(labelCol="Days late", predictionCol="prediction_delay", metricName="rmse", weightCol='weight')
rmse_delay = regression_evaluator.evaluate(predictions_delay)
print(f"RMSE for delay prediction: {rmse_delay}")

r2 = regression_evaluator.setMetricName('r2').evaluate(predictions_delay)
print(f"R2 for delay prediction: {r2}")

# Days Late:  mean = 1.62, std = 0.88, range = [1, 4]